# PharmaSign: تحويل نص الصيدلي إلى Gloss Tokens
هذا الدفتر يقرأ ملف Excel اسمه `Instructions.xlsx` يحتوي عمود **كلام الصيدلي** ويحوّله إلى:
- `pharmasign_text2gloss.csv`
- `pharmasign_text2gloss.json`

> ملاحظة: كود التحميل من Colab تم استبداله بحفظ ملفات محلياً (مناسب لـ GitHub).

In [ ]:
# (اختياري) ثبّت الحزم مرة واحدة إذا ما كانت موجودة
# !pip -q install pandas openpyxl

In [ ]:
import re
import pandas as pd
from collections import Counter

In [ ]:
# عدّل المسار إذا لازم
XLSX_PATH = "Instructions.xlsx"

df = pd.read_excel(XLSX_PATH)
print("Columns:", df.columns.tolist())

# العمود الأساسي
texts = df["كلام الصيدلي"].astype(str).fillna("").tolist()
print("Rows:", len(texts))
texts[:3]

In [ ]:
# Lexicon (قائمة كلمات الـ gloss المسموح فيها)
LEXICON = {
  # STRUCTURE
  "START-INSTRUCTION","END-INSTRUCTION","IMPORTANT","EMPHASIS",
  "DRUG","DOSE","TAKE","APPLY","USE","DURATION","WARNING","IF","THEN","STOP",
  "CALL","FOLLOW","MAX","ONLY","NOT","EVERY","UNTIL","DAILY","SAME-TIME",
  "CONTINUE","REDUCE","INCREASE","AVOID","DONT","FINISH",

  # UNITS / FORMS
  "MG","ML","DROP","TABLET","CAPSULE","PUFF","SACHET","GRAM","UNIT",
  "HOUR","DAY","WEEK","MONTH","TIMES-DAY","ONCE","TWICE","THREE-TIMES",
  "FULL","THIN","ALL","LOT","BOX",

  # TIME / FOOD
  "MORNING","NOON","NIGHT","BEFORE-SLEEP",
  "BEFORE-FOOD","AFTER-FOOD","WITH-FOOD","BEFORE","AFTER","WITH","IMMEDIATE","AFTER-OPEN",

  # ROUTE / FORM
  "ORAL","SKIN","EYE","EAR","NOSE","CHEST",
  "SYRUP","CREAM","OINTMENT","POWDER","INHALER","DROP-FORM",

  # PRN / CONDITIONS
  "IF-NEEDED","AS-NEEDED","IF-FEVER","IF-PAIN","IF-BREATH-SHORT",

  # WARNINGS / SIDE EFFECTS
  "ALLERGY","PENICILLIN","OVERDOSE","SIDE-EFFECT",
  "DROWSY","DIZZY","SWEAT","COUGH","MUSCLE-PAIN","NAUSEA","VOMIT","DIARRHEA",
  "RASH","SWELL","BREATH-PROBLEM","CHEST-PAIN","SEVERE","IMMEDIATE","SUDDEN",
  "LOW-SUGAR","HIGH-SUGAR","BLOOD-SUGAR",
  "GRAPEFRUIT",

  # ACTIONS
  "SHAKE","BREATH","DEEP","HOLD","WAIT","RINSE","WATER","SWALLOW",
  "DISSOLVE","DRINK","CHECK","MEASURE","THROW","TOUCH","PULL","HEAD-BACK",
  "INJECT","INSERT","MIX","STORE","KEEP",

  # FOLLOW-UP
  "DOCTOR","NO-IMPROVE","WORSE","REGULAR","EMERGENCY","NOT-DAILY","LONG-USE",

  # DISEASE / POP
  "CHILD","ADULT","ELDERLY","PREGNANT","BREASTFEED",
  "DIABETES","BLOOD-PRESSURE","ASTHMA","HEART",
  "SHAKING","CONFUSION","FAINT",

  # STORAGE
  "FRIDGE","ROOM-TEMP","SUN","DRY","CLOSE","AWAY-CHILD",
  "EXPIRE","COLOR-CHANGE","SMELL-CHANGE","DAMAGED",
}

len(LEXICON)

In [ ]:
def normalize_ar(text: str) -> str:
    t = str(text).strip()
    t = re.sub(r"[“”"']", "", t)
    t = re.sub(r"\s+", " ", t)
    # توحيد أحرف شائعة
    t = t.replace("أ","ا").replace("إ","ا").replace("آ","ا")
    t = t.replace("ة","ه")  # اختياري
    return t

# اختبار سريع
normalize_ar("لازم تاخد كبسولة وحدة كل 12 ساعة، يعني حبة الصبح وحبة المسا")

In [ ]:
def extract_slots(text: str) -> dict:
    t = normalize_ar(text)

    slots = {
        "form": None,                     # CAPSULE/TABLET/SYRUP/INHALER/CREAM/...
        "dose_value": None,               # رقم
        "dose_unit": None,                # MG/ML/DROP/PUFF/...
        "frequency_interval_hours": None, # كل X ساعة
        "frequency_per_day": None,        # X مرات باليوم
        "timing": [],                     # MORNING/NIGHT...
        "food_relation": None,            # AFTER-FOOD/BEFORE-FOOD/WITH-FOOD
        "duration_value": None,           # رقم
        "duration_unit": None,            # DAY/WEEK/MONTH
        "max_daily": None,                # حد أقصى
        "warnings": [],                   # tokens
        "instructions": [],               # actions: SHAKE, RINSE...
        "conditional": None,              # IF-FEVER / IF-PAIN / IF-BREATH-SHORT
        "notes": []                       # free
    }

    # frequency per day (مرات/باليوم)
    m = re.search(r"(\d+)\s*مر(ه|ات)\s*(بال|في)\s*اليوم", t)
    if m:
        slots["frequency_per_day"] = int(m.group(1))

    if "مره باليوم" in t or "مرة باليوم" in t:
        slots["frequency_per_day"] = 1
    if "مرتين باليوم" in t:
        slots["frequency_per_day"] = 2
    if "ثلاث مرات باليوم" in t or "3 مرات باليوم" in t:
        slots["frequency_per_day"] = 3

    # form detection (بدائي)
    if "كبسول" in t:
        slots["form"] = "CAPSULE"
    elif "حبه" in t or "قرص" in t:
        slots["form"] = "TABLET"
    elif "شراب" in t:
        slots["form"] = "SYRUP"
    elif "بخاخ" in t:
        slots["form"] = "INHALER"
    elif "كريم" in t:
        slots["form"] = "CREAM"
    elif "مرهم" in t:
        slots["form"] = "OINTMENT"
    elif "بودره" in t or "كيس" in t:
        slots["form"] = "POWDER"
    elif "قطره" in t or "نقط" in t:
        slots["form"] = "DROP-FORM"

    # food relation
    if "بعد الاكل" in t or "بعد الطعام" in t:
        slots["food_relation"] = "AFTER-FOOD"
    elif "قبل الاكل" in t or "قبل الطعام" in t:
        slots["food_relation"] = "BEFORE-FOOD"
    elif "مع الاكل" in t or "مع الطعام" in t:
        slots["food_relation"] = "WITH-FOOD"

    # timing
    if "الصبح" in t:
        slots["timing"].append("MORNING")
    if "الظهر" in t:
        slots["timing"].append("NOON")
    if "المسا" in t or "المساء" in t or "ليل" in t:
        slots["timing"].append("NIGHT")
    if "قبل النوم" in t:
        slots["timing"].append("BEFORE-SLEEP")

    # interval hours: كل 12 ساعة / كل 6 ساعات / كل 8 ساعات
    m = re.search(r"كل\s*(\d+)\s*ساع", t)
    if m:
        slots["frequency_interval_hours"] = int(m.group(1))

    # dose patterns: "5 مل" / "3 نقط" / "بختين"
    m = re.search(r"(\d+)\s*مل", t)
    if m:
        slots["dose_value"] = int(m.group(1))
        slots["dose_unit"] = "ML"

    m = re.search(r"(\d+)\s*نقط", t)
    if m:
        slots["dose_value"] = int(m.group(1))
        slots["dose_unit"] = "DROP"

    if "بختين" in t or "بختين" in t:
        slots["dose_value"] = 2
        slots["dose_unit"] = "PUFF"

    # max daily: "اكثر من اربع جرعات باليوم" أو "اكثر من 4"
    m = re.search(r"اكثر من\s*(\d+)", t)
    if m:
        slots["max_daily"] = int(m.group(1))
    if "اربع جرعات" in t and slots["max_daily"] is None:
        slots["max_daily"] = 4

    # conditional (بدائي)
    if ("وقت اللزوم" in t or "عند اللزوم" in t):
        if "حراره" in t:
            slots["conditional"] = "IF-FEVER"
        elif "الم" in t:
            slots["conditional"] = "IF-PAIN"
    if "ضيقه نفس" in t:
        slots["conditional"] = "IF-BREATH-SHORT"

    # actions / instructions
    if "خض" in t or "رج" in t:
        slots["instructions"].append("SHAKE")
    if "تمضمض" in t:
        slots["instructions"] += ["RINSE", "WATER"]
    if "تذوب" in t or "ذوب" in t:
        slots["instructions"].append("DISSOLVE")
    if "اشرب" in t:
        slots["instructions"].append("DRINK")
    if "بلع" in t:
        slots["instructions"].append("SWALLOW")
    if "سرنغ" in t or "سرنجه" in t:
        slots["instructions"].append("MEASURE")

    # warnings (keywords)
    if "حساسيه" in t:
        slots["warnings"].append("ALLERGY")
    if "بنسلين" in t:
        slots["warnings"].append("PENICILLIN")
    if "جريب فروت" in t:
        slots["warnings"].append("GRAPEFRUIT")
    if "دوخه" in t:
        slots["warnings"].append("DIZZY")
    if "سعال" in t:
        slots["warnings"].append("COUGH")

    return slots

# مثال سريع
extract_slots("خد كبسولة بعد الاكل مرتين باليوم")

In [ ]:
def make_gloss(slots: dict, lexicon: set[str] = LEXICON, warn_unknown: bool = False) -> list[str]:
    g = ["START-INSTRUCTION", "DRUG"]

    # FORM
    if slots.get("form"):
        g.append(slots["form"])

    # DOSE
    if slots.get("dose_value") is not None and slots.get("dose_unit"):
        g += ["DOSE", str(slots["dose_value"]), slots["dose_unit"]]

    # TAKE/APPLY based on form
    if slots.get("form") in {"CREAM","OINTMENT"}:
        g.append("APPLY")
    else:
        g.append("TAKE")

    # Interval
    if slots.get("frequency_interval_hours"):
        g += ["EVERY", str(slots["frequency_interval_hours"]), "HOUR"]

    # frequency_per_day -> TIMES-DAY (اختياري)
    if slots.get("frequency_per_day") is not None:
        g += ["TIMES-DAY", str(slots["frequency_per_day"])]

    # Timing
    for t in slots.get("timing", []):
        g.append(t)

    # Food relation
    if slots.get("food_relation"):
        g.append(slots["food_relation"])

    # Actions
    for act in slots.get("instructions", []):
        g.append(act)

    # Max daily
    if slots.get("max_daily") is not None:
        g += ["MAX", str(slots["max_daily"]), "TIMES-DAY"]

    # Warnings
    if slots.get("warnings"):
        g.append("WARNING")
        if "ALLERGY" in slots["warnings"] and "PENICILLIN" in slots["warnings"]:
            g += ["IF", "PENICILLIN", "ALLERGY", "DONT", "TAKE"]
        else:
            g += slots["warnings"]

    # Conditional
    if slots.get("conditional"):
        g += ["IF", slots["conditional"]]

    g.append("END-INSTRUCTION")

    if warn_unknown:
        for tok in g:
            if tok.isdigit():
                continue
            if tok not in lexicon:
                print("⚠️ Unknown token:", tok)

    return g

make_gloss(extract_slots("خد 5 مل شراب كل 8 ساعات بعد الاكل"))

In [ ]:
# بناء الداتا النهائية
rows = []
for i, text in enumerate(texts, 1):
    slots = extract_slots(text)
    gloss = make_gloss(slots)
    rows.append({
        "row_id": i,
        "kalam_al_saydali": text,
        "gloss_tokens": gloss,
        "gloss_text": " ".join(gloss),
        "slots": slots,
    })

out = pd.DataFrame(rows)
out[["row_id","kalam_al_saydali","gloss_text"]].head(10)

In [ ]:
# تقرير عن tokens غير موجودة بالـ lexicon
def list_unknown_tokens(gloss_tokens, lexicon):
    u = []
    for tok in gloss_tokens:
        if str(tok).isdigit():
            continue
        if tok not in lexicon:
            u.append(tok)
    return u

out["unknown_tokens"] = out["gloss_tokens"].apply(lambda g: list_unknown_tokens(g, LEXICON))
out["unknown_count"] = out["unknown_tokens"].apply(len)

cnt = Counter([t for lst in out["unknown_tokens"] for t in lst])
cnt.most_common(50)

In [ ]:
# (اختياري) إذا بدك توسّع lexicon بالتجريب
# for tok, _ in cnt.most_common():
#     LEXICON.add(tok)
# print("LEXICON size:", len(LEXICON))

In [ ]:
# حفظ المخرجات (مناسب لـ GitHub)
out.to_csv("pharmasign_text2gloss.csv", index=False, encoding="utf-8-sig")
out.to_json("pharmasign_text2gloss.json", orient="records", force_ascii=False, indent=2)
out.to_excel("pharmasign_text2gloss.xlsx", index=False)

["pharmasign_text2gloss.csv","pharmasign_text2gloss.json","pharmasign_text2gloss.xlsx"]

## (اختياري) تقييم سريع
إذا ملف الـ Excel فيه أعمدة gold مثل `gold_form`, `gold_dose_value`, ... فيك تشغّل التقييم التالي.

In [ ]:
# تشغيل التقييم فقط إذا الأعمدة موجودة
gold_cols = {"gold_form","gold_dose_value","gold_dose_unit","gold_every_hours","gold_food_relation","gold_max_daily","gold_conditional"}
if gold_cols.issubset(set(df.columns)):
    eval_df = df.sample(30, random_state=42).copy()
    eval_df["text"] = eval_df["كلام الصيدلي"].astype(str).fillna("")

    pred_slots = eval_df["text"].apply(extract_slots)
    eval_df["pred_form"] = pred_slots.apply(lambda d: d.get("form"))
    eval_df["pred_dose_value"] = pred_slots.apply(lambda d: d.get("dose_value"))
    eval_df["pred_dose_unit"] = pred_slots.apply(lambda d: d.get("dose_unit"))
    eval_df["pred_every_hours"] = pred_slots.apply(lambda d: d.get("frequency_interval_hours"))
    eval_df["pred_food_relation"] = pred_slots.apply(lambda d: d.get("food_relation"))
    eval_df["pred_max_daily"] = pred_slots.apply(lambda d: d.get("max_daily"))
    eval_df["pred_conditional"] = pred_slots.apply(lambda d: d.get("conditional"))

    def acc(col_pred, col_gold):
        m = eval_df[col_gold].notna()
        if m.sum() == 0:
            return None
        return (eval_df.loc[m, col_pred] == eval_df.loc[m, col_gold]).mean()

    metrics = {
      "form_acc": acc("pred_form", "gold_form"),
      "dose_value_acc": acc("pred_dose_value", "gold_dose_value"),
      "dose_unit_acc": acc("pred_dose_unit", "gold_dose_unit"),
      "every_hours_acc": acc("pred_every_hours", "gold_every_hours"),
      "food_relation_acc": acc("pred_food_relation", "gold_food_relation"),
      "max_daily_acc": acc("pred_max_daily", "gold_max_daily"),
      "conditional_acc": acc("pred_conditional", "gold_conditional"),
    }
    metrics
else:
    print("Gold columns not found. Skip evaluation.")